# SpaceX Falcon 9 First Stage Landing Prediction

Machine learning project from the IBM Data Science Professional Certificate.
The goal is to predict whether the Falcon 9 first stage will successfully land.

## Objectives

- Load and prepare the SpaceX dataset
- Standardize the features
- Split data into training and test sets
- Tune Logistic Regression, SVM, Decision Tree, and KNN models
- Compare model performance using test accuracy and confusion matrices

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn import preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

## Load the data

In [ ]:
data = pd.read_csv(
    'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_2.csv'
)

X = pd.read_csv(
    'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_3.csv'
)

display(data.head())
display(X.head())

## Prepare the target variable

In [ ]:
Y = data['Class'].to_numpy()
print('Target shape:', Y.shape)

## Standardize the features

In [ ]:
transform = preprocessing.StandardScaler()
X = transform.fit_transform(X)
print('Feature matrix shape:', X.shape)

## Train-test split

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=2
)

print('Training samples:', len(X_train))
print('Test samples:', len(X_test))

## Helper function: confusion matrix

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.xlabel('Predicted labels')
    plt.ylabel('True labels')
    plt.title(title)
    plt.show()

## 1. Logistic Regression

In [ ]:
parameters = {
    'C': [0.01, 0.1, 1],
    'penalty': ['l2'],
    'solver': ['lbfgs']
}

lr = LogisticRegression(max_iter=1000)
logreg_cv = GridSearchCV(lr, parameters, cv=10)
logreg_cv.fit(X_train, Y_train)

print('Best parameters:', logreg_cv.best_params_)
print('Validation accuracy:', logreg_cv.best_score_)
lr_pred = logreg_cv.predict(X_test)
lr_accuracy = accuracy_score(Y_test, lr_pred)
print('Test accuracy:', lr_accuracy)
plot_confusion_matrix(Y_test, lr_pred, 'Logistic Regression')

## 2. Support Vector Machine (SVM)

In [ ]:
parameters = {
    'kernel': ['linear', 'rbf', 'poly', 'sigmoid'],
    'C': np.logspace(-3, 3, 5),
    'gamma': np.logspace(-3, 3, 5)
}

svm = SVC()
svm_cv = GridSearchCV(svm, parameters, cv=10)
svm_cv.fit(X_train, Y_train)

print('Best parameters:', svm_cv.best_params_)
print('Validation accuracy:', svm_cv.best_score_)
svm_pred = svm_cv.predict(X_test)
svm_accuracy = accuracy_score(Y_test, svm_pred)
print('Test accuracy:', svm_accuracy)
plot_confusion_matrix(Y_test, svm_pred, 'Support Vector Machine')

## 3. Decision Tree

In [ ]:
parameters = {
    'criterion': ['gini', 'entropy'],
    'splitter': ['best', 'random'],
    'max_depth': [2, 4, 6, 8, 10],
    'min_samples_leaf': [1, 2, 4],
    'min_samples_split': [2, 5, 10]
}

tree = DecisionTreeClassifier(random_state=42)
tree_cv = GridSearchCV(tree, parameters, cv=10)
tree_cv.fit(X_train, Y_train)

print('Best parameters:', tree_cv.best_params_)
print('Validation accuracy:', tree_cv.best_score_)
tree_pred = tree_cv.predict(X_test)
tree_accuracy = accuracy_score(Y_test, tree_pred)
print('Test accuracy:', tree_accuracy)
plot_confusion_matrix(Y_test, tree_pred, 'Decision Tree')

## 4. K-Nearest Neighbors (KNN)

In [ ]:
parameters = {
    'n_neighbors': list(range(1, 11)),
    'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
    'p': [1, 2]
}

knn = KNeighborsClassifier()
knn_cv = GridSearchCV(knn, parameters, cv=10)
knn_cv.fit(X_train, Y_train)

print('Best parameters:', knn_cv.best_params_)
print('Validation accuracy:', knn_cv.best_score_)
knn_pred = knn_cv.predict(X_test)
knn_accuracy = accuracy_score(Y_test, knn_pred)
print('Test accuracy:', knn_accuracy)
plot_confusion_matrix(Y_test, knn_pred, 'K-Nearest Neighbors')

## Model Comparison

In [ ]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'SVM', 'Decision Tree', 'KNN'],
    'Test Accuracy': [lr_accuracy, svm_accuracy, tree_accuracy, knn_accuracy]
}).sort_values('Test Accuracy', ascending=False)

display(results)
print('Best-performing model:', results.iloc[0]['Model'])

## Conclusion

Several classification algorithms were trained and evaluated to predict Falcon 9 first-stage landing success. Hyperparameter tuning was performed using GridSearchCV, and the models were compared using test-set accuracy and confusion matrices. The model with the highest test accuracy is selected as the best-performing approach for this dataset.